<a href="https://colab.research.google.com/github/akshin-kuvshin/numerical-methods/blob/main/lab9/lab9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа №9. Вычисление интеграла методом Чебышева

## Выполнил: Данила Аксёнов, гр. 932420

## Вариант: 1

## **Задание 14**

* Вычислить $ \int_{a}^{b} f(x) \, dx $ с точностью до $ \epsilon = 0.5 * 10^{-5} $ по формуле Чебышева;
* Вывести значение интеграла и число узлов, которое потребовалось для вычисления интеграла с заданной точностью.

In [10]:
import numpy as np
from matplotlib import pyplot as plt
print('Setup completed')

Setup completed


### Исходные данные:

In [11]:
a = 2
b = 3
f = lambda x: np.sqrt(x ** 2 + 2 * x + np.log10(x))
eps = 0.5 * 10 ** -5

### Решение:

Для вычисления интеграла я решил использовать следующую формулу, использующую $ mn $ узлов:

$$ I = \int_{a}^{b} f(x) \, dx = \frac{b - a}{mn} \sum_{i = 1}^{m} \sum_{k = 1}^{n} f \left( a + \frac{b - a}{2m} (t_k + 2i - 1) \right) $$

(см. учебник, стр. 164).

Принцип точно такой же, как и в предыдущей лабораторной работе (предыдущем задании): отрезок интегрирования $ [a; b] $ делится на $ m $ равных частей, на каждой из которых мы вычисляем интеграл по _формуле Чебышева, квадратурному правилу с **равными коэффициентами**_ $ - $ это ключевое отличие формулы Чебышева от формулы Гаусса. $ n $ узлов $ t_k $ внутри каждой из $ m $ частей находятся из того условия, чтобы формула Чебышева давала точный результат для функций вида $ f(t) = t, t^2, ..., t^n $. Для них, опять же, _нет общей формулы_, но это не является проблемой, т.к., согласно _теореме Бернштейна_, для $ n = 8 $ и для всех $ n \gt 9 $ среди искомых узлов $ t_k $ **присутствуют комплексные** $ \Rightarrow $ **формула Чебышева применима только для** $ \mathbf{n = 1, 2, ..., 7, 9} $.

Для максимальной точности зафиксируем максимально возможное допустимое $ n = 9 $ (это, конечно, при постоянной мощности "внешнего" разбиения $ m $ увеличивает общее количество узлов, в которых вычисляется значение функции, но, по моему мнению, на данное различие в константу можно закрыть глаза, т.к. при любом выбранном и зафиксированном $ n $ величину $ mn $ можно оценить, как $ \mathcal{O}(m) $). К сожалению, в нашем учебнике (Смагин, Решетникова) нет приближенных значений узлов для этого "внутреннего" разбиения, поэтому я воспользовался сторонним учебником: _Крылов, Бобков, Монастырный $ - $ "Вычислительные методы высшей математики", том 1_ (https://drive.google.com/file/d/1kk6V_CYEcsbeB3MKvcWVWyJENJxudoa7/view?usp=drive_link $ - $ загрузил его к себе на Диск; этот же учебник пригодится нам в дальнейшем), $ - $ в котором нужные мне данные приведены на странице 468.

Ввиду фиксации $ n $ и его общей ограниченности сверху, заданной точности будет добиваться путём увеличения мощности "внешнего" разбиения $ m $, как мы делали это ранее.

---

Для нахождения порядка чётности формулы Чебышева, снова обратимся к учебнику Крылова, Бобкова и Монастырного: на странице 467 утверждается: _"При чётном $ n $... правило... будет точным для многочленов степени $ n + 1 $, а не только степени $ n $, как предусматривается в постановке задачи"_. Это означает, что при нечётном $ n $ формула Чебышева будет давать точный результат для функций вида $ f(t) = t, t^2, ..., t^n $, а при чётном $ - $ для функций вида $ f(t) = t, t^2, ..., t^n, t^{n + 1} $ $ \Rightarrow $ по определению:

$$ \mathbf{\text{Степень точности } d = \begin{cases} n, & \text{если } n \text{ нечётное}, \\ n + 1, & \text{если } n \text{ чётное}. \end{cases}} $$

По определению, _**порядок точности** $ - $ это такое число $ \mathbf{p} $, что остаточный член $ R(h) $ имеет асимптотику $ \mathcal{O}(h^p) $_

$$ \Downarrow $$

$$ \mathbf{\text{Порядок точности } p = d + 1 = \begin{cases} n + 1, & \text{если } n \text{ нечётное}, \\ n + 2, & \text{если } n \text{ чётное}. \end{cases}} $$

Как мы уже оговорили ранее, порядок точности сохраняется при суммировании по всем $ m $ частям "внешнего" разбиения. Таким образом, найдя его (порядок точности), мы имеем полное право использовать метод Рунге для остановки:

$$ \frac{\left\lvert I_h - I_{\frac{h}{2}} \right\rvert}{2^p - 1} \le \epsilon \quad \Rightarrow \quad \text{прекращаем итерации}, $$

где $ h $ и $ \frac{h}{2} $ $ - $ шаги "внешнего" разбиения для мощностей $ m $ и $ 2m $ соответственно. В конце, как и множество раз ранее, сделаем поправку по методу Рунге (_поправку/экстраполяцию Ричардсона_):

$$ I \approx I_{\frac{h}{2}} + \frac{I_h - I_{\frac{h}{2}}}{1 - 2^p}. $$

К сожалению, при вычислении значения определённого интеграла _"расширенной"_ формулой Чебышева, как и в случае с _"расширенной"_ формулой Гаусса, "внешние" разбиения мощностей $ m $ и $ 2m $, в общем случае, не имеют общих узлов, в которых для них предстоит вычислять значение функции $ f $, поэтому каждый раз нам придётся пересчитывать значение интеграла "с нуля", т.е. заново.

---

<u>P.S.</u>: в учебнике узлы формулы Чебышева обозначены как $ x_k $, но я, как и прежде, решил поменять их обозначение на $ t_k $, чтобы не путать с независимой переменной $ x $, т.к., в общем случае, $ x \in [a; b] $, в то время как $ t_k \in [-1; 1] $.

### Дополнительные исходные данные:

In [12]:
n = 9
t = np.array([-0.9115893007, -0.6010186554, -0.5287617831, -0.1679061842, 0.0000000000, 0.1679061842, 0.5287617831, 0.6010186554, 0.9115893007])

d = n + (1 - n % 2)
p = d + 1

# заранее вычислим выражение (знаменатель), использующееся(-ийся) в методе Рунге
# (как для остановки, так и для поправки), чтобы не пересчитывать его на каждой итерации
q = 2 ** p - 1

print('Additional setup completed')

Additional setup completed


In [13]:
# функция, вычисляющая значение интеграла для заданной мощности "внешнего" разбиения m
def calc_I(m):
    c = (b - a) / m

    I = 0.0
    for i in range(1, m + 1):
        for k in range(n):
            I += f(a + c / 2 * (t[k] + 2 * i - 1))
    I *= c / n
    return I

In [14]:
m = 1
Im = calc_I(m)
while True:
    m2 = m * 2
    Im2 = calc_I(m2)
    if abs(Im - Im2) / q <= eps:
        break
    m = m2
    Im = Im2
print('done.')

done.


In [15]:
q_neg = -q
I = Im2 + (Im - Im2) / q_neg
I, m2, m2 * n

(np.float64(3.4111713027476127), 2, 18)

In [16]:
abs(Im - Im2)

np.float64(2.176969715605992e-11)

Следующий результат был получен с использованием _"расширенной"_ формулы Чебышева: $ \mathbf{\int_a^b f(x)\,dx = \int_2^3 \sqrt{x^2 + 2x + \lg(x)} \, dx \approx 3.41117} $, $ - $ он (результат) совпадает с результатом, полученным с использованием "расширенной" формулы Гаусса, вплоть до 5-ого знака после запятой и с результатами, полученными в лабораторной работе (задании) про методы Ньютона-Котеса, вплоть до 4-ого знака после запятой, что подверждает корректность и успешность реализации.

Для получения результата с точностью до $ \epsilon = 0.5 * 10^{-5} $ потребовалось всего $ \mathbf{mn = 2 * 9 = 18} $ **узлов**, в которых оказалось необходимым посчитать значение функции $ f $, что, опять же, говорит о _высокой точности формулы Чебышева_ $ - $ _квадратурного правила с **равными коэффициентами**_. И вновь стоит отметить, что мощность "внешнего" разбиения $ m = 1 $ также удовлетворяет все наши потребности, т.к. значения интегралов при $ m = 1 $ и при $ m = 2 $ отличаются друг от друга лишь в 11-ом знаке после запятой, так что можно было обойтись и $ 1 * 9 = 9 $ узлами.